# 08. Successor Representations Forward Backward Zero Shot RL and DIAYN

- **Category:** 08_learning_paradigms_rl_and_alignment / 02_reinforcement_learning_and_world_models
- **Lineage & Citations:** Dayan (1993 Successor Representation); Barreto et al. (2017, 2018 Successor Features & Generalized Policy Improvement); Eysenbach et al. (2018 Diversity Is All You Need / DIAYN); Touati & Ollivier (2021 Forward-Backward Zero-Shot RL).
- **Purpose:** First-principles mathematical implementation and empirical verification of task-agnostic representation learning: Successor Features (SF) temporal difference updates, transfer via Generalized Policy Improvement (GPI), Forward-Backward (FB) zero-shot bilinear value evaluation, and DIAYN mutual information skill discovery.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [07. Multi-Agent RL Qmix Mappo CFR and Agentic Reasoning Loops](07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.ipynb) $\to$ `🔗 Downstream Syntheses:` [01. Supervised Finetuning Chat Masks Neftune and Sequence Packing](../03_llm_alignment_and_reasoning/01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Never use Successor Features when task rewards cannot be accurately factored into a linear inner product $\boldsymbol{\phi}(s)^T \mathbf{w}$; non-linear reward structures fundamentally violate the bilinear value decomposition $Q(s, a) = \boldsymbol{\psi}(s, a)^T \mathbf{w}$. In DIAYN, never pass raw global coordinates to the skill discriminator; policies will collapse into static posture freezing in distinct corners of the state space without discovering meaningful motion.

## Mathematical Foundations & Governing Equations

### 1. Successor Features TD Error
$$
\boldsymbol{\delta}_{\text{SF}} = \boldsymbol{\phi}(s) + \gamma (1 - d) \boldsymbol{\psi}(s', a') - \boldsymbol{\psi}(s, a)
$$

### 2. Generalized Policy Improvement (GPI)
$$
Q_{\text{GPI}}(s, a) = \max_{i=1,\dots,K} \boldsymbol{\psi}^{\pi_i}(s, a)^T \mathbf{w}
$$

### 3. Forward-Backward (FB) Bilinear Zero-Shot Value
$$
Q_{\text{FB}}(s, a, z) = \sum_{d=1}^D F(s, a, z)_d \cdot z_d
$$

### 4. DIAYN Skill Intrinsic Reward & Discriminator Loss
$$
r_{\text{DIAYN}}(s, z) = \ln q_\phi(z \mid s) - \ln p(z) = \ln q_\phi(z \mid s) + \ln K
$$
$$
\mathcal{L}_{\text{disc}}(\phi) = -\frac{1}{B} \sum_{b=1}^B \ln q_\phi(z_b \mid s_b)
$$

### Asymptotic Time & Memory Complexity
- Successor Features TD Error: $\mathcal{O}(B \cdot D)$ time and memory.
- GPI Candidate Envelope: $\mathcal{O}(B \cdot K \cdot D)$ time and memory.
- Forward-Backward Bilinear Value: $\mathcal{O}(B \cdot D)$ time and memory.
- DIAYN Mutual Information Reward: $\mathcal{O}(B \cdot K)$ time and memory.

In [ ]:
# Setup and seed verification environment
import math
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/08_successor_representations_forward_backward_zero_shot_rl_and_diayn.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def successor_features_td_error(
    psi_current: torch.Tensor,
    features: torch.Tensor,
    psi_next: torch.Tensor,
    dones: torch.Tensor,
    gamma: float = 0.99,
) -> torch.Tensor:
    non_terminal = (1.0 - dones).unsqueeze(-1)
    target = features + gamma * non_terminal * psi_next
    return target - psi_current

def gpi_transfer_value(
    psi_policies: torch.Tensor,
    reward_weights: torch.Tensor,
) -> torch.Tensor:
    q_candidates = torch.matmul(psi_policies, reward_weights)
    return torch.max(q_candidates, dim=-1)[0]

def fb_zero_shot_value(
    forward_map: torch.Tensor,
    task_embedding: torch.Tensor,
) -> torch.Tensor:
    return torch.sum(forward_map * task_embedding, dim=-1)

def diayn_skill_reward_and_loss(
    logits: torch.Tensor,
    skill_labels: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    log_probs = F.log_softmax(logits, dim=-1)
    b = logits.shape[0]
    num_skills = logits.shape[-1]
    log_prior = -math.log(num_skills)
    skill_log_probs = log_probs.gather(dim=-1, index=skill_labels.unsqueeze(-1)).squeeze(-1)
    rewards = skill_log_probs - log_prior
    loss = -torch.mean(skill_log_probs)
    return rewards, loss

In [ ]:
# Multi-track numerical parity assertions
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-5, f"Linear anchor parity error: {err_anchor:.2e}"

sf_td = successor_features_td_error(weights["psi_current"], weights["features"], weights["psi_next"], weights["dones"], gamma=0.99)
err_sf = float(torch.max(torch.abs(sf_td - weights["sf_td_error_ref"])))
assert err_sf < 1e-5, f"Successor features TD error parity error: {err_sf:.2e}"

gpi_val = gpi_transfer_value(weights["psi_policies"], weights["reward_weights"])
err_gpi = float(torch.max(torch.abs(gpi_val - weights["gpi_values_ref"])))
assert err_gpi < 1e-5, f"GPI transfer value parity error: {err_gpi:.2e}"

fb_val = fb_zero_shot_value(weights["forward_map"], weights["task_embedding"])
err_fb = float(torch.max(torch.abs(fb_val - weights["fb_values_ref"])))
assert err_fb < 1e-5, f"FB zero-shot value parity error: {err_fb:.2e}"

diayn_r, diayn_l = diayn_skill_reward_and_loss(weights["logits"], weights["skill_labels"])
err_diayn_r = float(torch.max(torch.abs(diayn_r - weights["diayn_rewards_ref"])))
err_diayn_l = float(torch.abs(diayn_l - weights["diayn_loss_ref"]))
assert err_diayn_r < 1e-5, f"DIAYN rewards parity error: {err_diayn_r:.2e}"
assert err_diayn_l < 1e-5, f"DIAYN loss parity error: {err_diayn_l:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_sf, err_gpi, err_fb, err_diayn_r, err_diayn_l):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_sf = benchmark_fn(
    lambda: successor_features_td_error(weights["psi_current"], weights["features"], weights["psi_next"], weights["dones"]),
    iters=100,
    name="Successor Features TD Error",
)
b_gpi = benchmark_fn(
    lambda: gpi_transfer_value(weights["psi_policies"], weights["reward_weights"]),
    iters=100,
    name="GPI Transfer Value",
)
b_fb = benchmark_fn(
    lambda: fb_zero_shot_value(weights["forward_map"], weights["task_embedding"]),
    iters=100,
    name="FB Zero-Shot Bilinear Value",
)
b_diayn = benchmark_fn(
    lambda: diayn_skill_reward_and_loss(weights["logits"], weights["skill_labels"]),
    iters=100,
    name="DIAYN Skill Reward & Loss",
)

print(format_benchmark_table([b_sf, b_gpi, b_fb, b_diayn]))

In [ ]:
# Failure Mode Contrast: Static Posture Freezing vs Dynamic Velocity Conditioning in DIAYN
# Suppose K=5 discrete skills, so prior p(z) = 0.20 and log(K) = log(5) = 1.6094
log_k = math.log(5)

# Case 1: Discriminator conditioned on raw coordinates (agent stays motionless at x=5.0)
# Discriminator easily overfits to static posture, predicting skill z=0 with 99% probability:
p_raw_static = 0.99
reward_raw_static = math.log(p_raw_static) + log_k

# Case 2: Discriminator conditioned on delta state / velocity coordinates (agent is motionless, v=0)
# When motionless (v=0), all skills look identical: discriminator can only predict uniform probability (p=1/K=0.20):
p_velocity_static = 0.20
reward_velocity_static = math.log(p_velocity_static) + log_k

print(f"Raw State DIAYN Static Pose Reward:       {reward_raw_static:.4f} (Spurious high reward rewards motionless posture!)")
print(f"Velocity-Conditioned DIAYN Static Reward: {reward_velocity_static:.4f} (Zero reward, forces active kinetic movement!)")

assert reward_raw_static > 1.5, f"Raw state static pose should produce high spurious reward (got {reward_raw_static})"
assert abs(reward_velocity_static - 0.0) < 1e-4, f"Velocity-conditioned static pose must give 0 reward (got {reward_velocity_static})"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Decoupled Representation & Reward**: Successor Features factor transition occupancies from rewards, enabling zero-interaction evaluation when objectives shift.
2. **GPI Envelope Guarantee**: Generalized Policy Improvement mathematically guarantees that the transferred policy weakly dominates all previous candidate policies.
3. **Universal Bilinear Dynamics (FB)**: Inner-product maps $Q(s, a, z) = \langle F, z \rangle$ extend zero-shot capability to continuous task spaces without explicit feature engineering.
4. **Information-Theoretic Exploration (DIAYN)**: Maximizing mutual information $\mathcal{I}(S; Z)$ discovers structured behavioral primitives without environmental supervision.

---
### Cross-Domain Generalisation Matrix
- **Multi-Task Robotics**: Successor features adapt robot arms to novel spatial end-effector targets in milliseconds.
- **Unsupervised Skill Bootstrapping**: DIAYN provides reusable locomotion options for downstream hierarchical controllers.
- **Dynamic Risk Portfolio Re-Optimization**: Forward-Backward representations re-evaluate optimal liquidation trajectories under shifting transaction fee structures.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [08_successor_representations_forward_backward_zero_shot_rl_and_diayn.md](../../../markdown/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/08_successor_representations_forward_backward_zero_shot_rl_and_diayn.md)
- **PyTorch 2.x Module:** [08_successor_representations_forward_backward_zero_shot_rl_and_diayn.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/08_successor_representations_forward_backward_zero_shot_rl_and_diayn.py)
- **Functional JAX Module:** [08_successor_representations_forward_backward_zero_shot_rl_and_diayn.py](../../../jax/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/08_successor_representations_forward_backward_zero_shot_rl_and_diayn.py)
- **Native Rust Candle Track:** [08_successor_representations_forward_backward_zero_shot_rl_and_diayn.rs](../../../candle/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/08_successor_representations_forward_backward_zero_shot_rl_and_diayn.rs)
- **Hardware Kernel (CPU / WGSL):** [08_successor_representations_forward_backward_zero_shot_rl_and_diayn.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/08_successor_representations_forward_backward_zero_shot_rl_and_diayn.rs)
- **Golden SafeTensors:** [08_successor_representations_forward_backward_zero_shot_rl_and_diayn.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/08_successor_representations_forward_backward_zero_shot_rl_and_diayn.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 07. Multi-Agent RL Qmix Mappo CFR and Agentic Reasoning Loops](07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 01. Supervised Finetuning Chat Masks Neftune and Sequence Packing](../03_llm_alignment_and_reasoning/01_supervised_finetuning_chat_masks_neftune_and_sequence_packing.ipynb) &rarr;